In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install sentence-transformers faiss-cpu python-docx -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.1 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer

In [5]:
PROJECT_DIR = "/content/drive/MyDrive/PrismHire"

candidates = []

with open(
    f"{PROJECT_DIR}/data/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/candidates.jsonl",
    "r"
) as f:

    for line in f:
        candidates.append(
            json.loads(line)
        )

In [6]:
def build_candidate_document(candidate):

    profile = candidate["profile"]

    skills = [
        s["name"]
        for s in candidate["skills"]
    ]

    career_text = []

    for job in candidate["career_history"]:

        text = (
            job["title"]
            + " "
            + job["industry"]
            + " "
            + job["description"]
        )

        career_text.append(text)

    document = f"""
    Headline:
    {profile['headline']}

    Summary:
    {profile['summary']}

    Current Role:
    {profile['current_title']}

    Industry:
    {profile['current_industry']}

    Skills:
    {' '.join(skills)}

    Career:
    {' '.join(career_text)}
    """

    return document

In [7]:
candidate_docs = []

for c in candidates:
    candidate_docs.append(
        build_candidate_document(c)
    )

len(candidate_docs)

100000

In [8]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
candidate_embeddings = model.encode(
    candidate_docs,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

In [10]:
candidate_embeddings.shape

(100000, 384)

In [11]:
np.save(
    f"{PROJECT_DIR}/models/candidate_embeddings.npy",
    candidate_embeddings
)